In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model
from langchain_teddynote import logging

from dotenv import load_dotenv

load_dotenv(override=True)

# langsmith 추적 설정
logging.langsmith("samsung_asset_ai_portal")

In [ ]:
from typing import List

def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)

In [ ]:
# Docling Loader - PDF를 마크다운으로 변환

import os
from pathlib import Path
from typing import Optional

# TESSDATA_PREFIX 환경 변수를 모듈 레벨에서 설정
# docling이 import될 때 tesserocr를 초기화할 수 있으므로 미리 설정
_tessdata_path = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
if not os.environ.get('TESSDATA_PREFIX'):
    os.environ['TESSDATA_PREFIX'] = _tessdata_path

# tesserocr를 직접 import하여 환경 변수를 확인하도록 강제
# docling이 내부적으로 tesserocr.get_languages()를 호출할 때 환경 변수를 읽을 수 있도록
try:
    import tesserocr
    # tesserocr를 초기화하여 환경 변수를 확인하도록 강제
    # get_languages()가 환경 변수를 읽지 못하는 문제를 해결하기 위해
    # PyTessBaseAPI를 사용하여 초기화
    _test_api = tesserocr.PyTessBaseAPI(path=_tessdata_path)
    _test_api.End()
except ImportError:
    # tesserocr가 설치되지 않은 경우 무시 (나중에 오류 처리됨)
    pass
except Exception:
    # 초기화 실패는 무시 (나중에 오류 처리됨)
    pass

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption

# Tesseract OCR과 CPU를 명시적으로 지정하는 설정
# PdfFormatOption의 pipeline_options를 통해 ThreadedPdfPipelineOptions 전달
# ThreadedPdfPipelineOptions의 ocr_options에 TesseractOcrOptions 지정
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    TesseractOcrOptions
)
from docling.datamodel.accelerator_options import AcceleratorOptions

# 암호화된 PDF를 다른 라이브러리로 읽어서 암호를 해제하고 임시 파일로 저장
import tempfile



def extract_text_from_pdf_with_docling(pdf_path: str, password: str = None) -> str:
    """
    Docling 라이브러리를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    Docling은 IBM에서 개발한 문서 변환 라이브러리로, 표, 레이아웃, 구조를 잘 보존하며
    마크다운으로 변환합니다.
    
    Tesseract OCR과 CPU 명시적 지정:
    - OCR 모델: Tesseract (PdfFormatOption의 pipeline_options를 통해 명시적으로 지정)
    - 가속기: CPU (AcceleratorOptions를 통해 명시적으로 지정)
    - 암호화된 PDF: pdfplumber로 텍스트와 표를 추출하여 마크다운으로 변환
    - 암호화되지 않은 PDF: docling으로 직접 마크다운 변환
    
    설정 방법:
    - PdfFormatOption의 pipeline_options 파라미터에 ThreadedPdfPipelineOptions 전달
    - ThreadedPdfPipelineOptions의 ocr_options에 TesseractOcrOptions 지정
    - ThreadedPdfPipelineOptions의 accelerator_options에 AcceleratorOptions(device='cpu') 지정
    - sys.platform 변경 없이 공식 API를 통해 명시적으로 설정
    
    주의사항:
    - TesseractOcrOptions를 사용하려면 tesserocr 라이브러리와 올바른 설정이 필요합니다.
    - Tesseract OCR 설정 오류 시 폴백 없이 오류가 발생하여 프로세스가 중지됩니다.
    - AcceleratorOptions(device='cpu')를 통해 CPU 사용이 명시적으로 지정됩니다.
    - TESSDATA_PREFIX 환경 변수가 올바르게 설정되어 있어야 합니다.
    
    Args:
        pdf_path: PDF 파일 경로 (상대 경로 또는 절대 경로)
        password: 암호화된 PDF의 비밀번호 (선택사항)
    
    Returns:
        마크다운 형식으로 변환된 문자열
    
    Raises:
        FileNotFoundError: PDF 파일을 찾을 수 없을 때
        ImportError: docling 라이브러리가 설치되지 않았을 때
        Exception: PDF를 읽을 수 없을 때
    """
    # TESSDATA_PREFIX 환경 변수 확인 및 설정
    # 모듈 레벨에서 이미 설정되었지만, 함수 내에서도 재확인
    tessdata_path = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
    if not os.environ.get('TESSDATA_PREFIX'):
        os.environ['TESSDATA_PREFIX'] = tessdata_path
    
    # 파일 경로 확인 및 절대 경로로 변환
    pdf_path = Path(pdf_path)
    if not pdf_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        pdf_path = project_root / pdf_path
    
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {pdf_path}")
    
    try:
        # 암호화된 PDF 처리: pdfplumber로 텍스트와 표를 추출하여 마크다운으로 변환
        if password:
            try:
                import pdfplumber
                
                markdown_parts = []
                with pdfplumber.open(str(pdf_path), password=password) as pdf:
                    for page_num, page in enumerate(pdf.pages, 1):
                        page_content = []
                        
                        # 표 추출 (표가 있으면 먼저 표를 추출)
                        tables = page.extract_tables()
                        if tables:
                            for table_idx, table in enumerate(tables):
                                if table:
                                    markdown_table = table_to_markdown(table)
                                    if markdown_table:
                                        page_content.append(markdown_table)
                                        page_content.append("")  # 표 다음에 빈 줄 추가
                        
                        # 텍스트 추출
                        text = page.extract_text()
                        if text:
                            page_content.append(text)
                        
                        if page_content:
                            markdown_parts.append("\n".join(page_content))
                
                return "\n\n".join(markdown_parts) if markdown_parts else ""
                
            except ImportError:
                raise ImportError(
                    "암호화된 PDF를 처리하기 위해 pdfplumber가 필요합니다.\n"
                    "설치 명령: pip install pdfplumber"
                )
            except Exception as e:
                # 암호화 관련 오류인지 확인
                error_msg = str(e).lower()
                if 'password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg:
                    raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
                raise
        
        # 암호화되지 않은 PDF: docling으로 직접 마크다운 변환
        # Docling 라이브러리 사용
        # Tesseract OCR과 CPU를 명시적으로 지정하여 사용
        
        # TESSDATA_PREFIX 환경 변수 설정 (tesserocr 사용 시 필요)
        tessdata_path = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
        if not os.environ.get('TESSDATA_PREFIX'):
            os.environ['TESSDATA_PREFIX'] = tessdata_path       
        
        # Tesseract OCR 옵션 생성 (실패 시 오류 발생, 폴백 없음)
        # TesseractOcrOptions를 사용하려면 tesserocr 라이브러리와 올바른 설정이 필요합니다.
        tesseract_ocr_opts = TesseractOcrOptions(
            lang=['eng', 'kor'],  # 영어와 한국어 지원
            bitmap_area_threshold=0.01,
            force_full_page_ocr=False,
            path=tessdata_path  # tessdata 경로 명시
        )
        
        # ThreadedPdfPipelineOptions 생성 (Tesseract OCR과 CPU 지정)
        threaded_pipeline_opts = ThreadedPdfPipelineOptions(
            ocr_options=tesseract_ocr_opts,
            accelerator_options=AcceleratorOptions(device='cpu')
        )
        
        # PdfFormatOption에 pipeline_options 전달
        pdf_option = PdfFormatOption(
            pipeline_options=threaded_pipeline_opts
        )
        
        converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={
                InputFormat.PDF: pdf_option
            }
        )
        
        print("ℹ️ Tesseract OCR과 CPU가 pipeline_options를 통해 명시적으로 지정되었습니다.")
        
        # converter.convert() 호출 전에 TESSDATA_PREFIX 환경 변수 재확인 및 설정
        # pipeline 초기화 시점에 tesserocr가 환경 변수를 확인하므로 명시적으로 설정
        if not os.environ.get('TESSDATA_PREFIX'):
            os.environ['TESSDATA_PREFIX'] = tessdata_path
        print(f"   TESSDATA_PREFIX: {os.environ.get('TESSDATA_PREFIX')}")

        # PDF 변환 (암호화되지 않은 PDF)
        # Tesseract OCR 실패 시 오류 발생 (폴백 없음)
        result = converter.convert(str(pdf_path))
        
        # 마크다운으로 내보내기
        markdown_content = result.document.export_to_markdown()
        
        return markdown_content
        
    except ImportError as import_err:
        # ImportError 처리 (Tesseract OCR 설정 오류 포함)
        error_msg = str(import_err)
        
        if 'docling' in error_msg.lower() and 'tesserocr' not in error_msg.lower():
            # docling import 실패
            raise ImportError(
                f"Docling 라이브러리가 설치되지 않았습니다.\n"
                f"오류 메시지: {error_msg}\n"
                f"설치 명령: pip install docling\n"
                f"또는: pip install 'docling[pdf]'"
            )
        elif 'tesserocr' in error_msg.lower():
            # Tesseract OCR 설정 오류 - 폴백 없이 오류 발생
            tessdata_path_for_error = os.environ.get('TESSDATA_PREFIX', '/opt/homebrew/share/tessdata')
            raise ImportError(
                f"Tesseract OCR 설정 오류가 발생했습니다.\n"
                f"오류 메시지: {error_msg}\n"
                f"\n"
                f"해결 방법:\n"
                f"1. tesserocr가 설치되어 있는지 확인: pip install tesserocr\n"
                f"2. TESSDATA_PREFIX 환경 변수가 올바르게 설정되어 있는지 확인\n"
                f"3. tessdata 경로에 언어 모델 파일이 있는지 확인: {tessdata_path_for_error}\n"
                f"4. 한글 언어 모델 확인: {tessdata_path_for_error}/kor.traineddata"
            )
        else:
            # 다른 ImportError는 그대로 전파
            raise
    except Exception as e:
        # 모든 오류를 명확하게 출력
        error_msg = str(e)
        error_type = type(e).__name__
        import traceback
        
        print(f"❌ Docling PDF 변환 오류 발생")
        print(f"   오류 유형: {error_type}")
        print(f"   오류 메시지: {error_msg}")
        print(f"\n   전체 스택 트레이스:")
        traceback.print_exc()
        
        # 암호화된 PDF 관련 오류인지 확인
        # password가 제공되었는데 오류가 발생하면 암호화된 PDF 처리 실패로 간주
        if password:
            # 암호화 관련 키워드 확인
            is_password_error = (
                'password' in error_msg.lower() or 
                'encrypted' in error_msg.lower() or 
                'incorrect password' in error_msg.lower() or
                'not valid' in error_msg.lower() or
                'inconsistent number of pages' in error_msg.lower() or
                error_type == 'ConversionError'
            )
            
            if is_password_error:
                raise ValueError(
                    f"Docling으로 암호화된 PDF를 처리할 수 없습니다.\n"
                    f"오류 유형: {error_type}\n"
                    f"오류 메시지: {error_msg}\n"
                    f"암호화된 PDF는 extract_text_from_pdf() 함수를 사용해주세요."
                )
        
        # 기타 오류
        raise Exception(
            f"PDF 변환 중 오류가 발생했습니다.\n"
            f"오류 유형: {error_type}\n"
            f"오류 메시지: {error_msg}"
        )
        
# 사용 예시
# pdf_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# try:
#     markdown_text = extract_text_from_pdf_with_docling(pdf_file_path)
#     print(f"✅ Docling으로 PDF 마크다운 변환 완료 ({len(markdown_text)} 문자)")
#     print("\n" + "="*80)
#     print("변환된 마크다운 (처음 1000자):")
#     print("="*80)
#     print(markdown_text[:1000])
#     if len(markdown_text) > 1000:
#         print(f"\n... (총 {len(markdown_text)} 문자 중 처음 1000자만 표시)")
# except Exception as e:
#     print(f"❌ 오류 발생: {e}")

In [ ]:
# excel loader - 모든 sheet 텍스트 추출

import os
from pathlib import Path
from typing import Dict, List

def extract_text_from_excel(excel_path: str) -> Dict[str, str]:
    """
    Excel 파일(.xlsx, .xls)에서 모든 시트의 텍스트를 추출하는 함수
    
    Args:
        excel_path: Excel 파일 경로 (상대 경로 또는 절대 경로)
    
    Returns:
        시트 이름을 키로 하고 추출된 텍스트를 값으로 하는 딕셔너리
    
    Raises:
        FileNotFoundError: Excel 파일을 찾을 수 없을 때
        ImportError: 필요한 Excel 라이브러리가 설치되지 않았을 때
    """
    # 파일 경로 확인 및 절대 경로로 변환
    excel_path = Path(excel_path)
    if not excel_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        excel_path = project_root / excel_path
    
    if not excel_path.exists():
        raise FileNotFoundError(f"Excel 파일을 찾을 수 없습니다: {excel_path}")
    
    print(f"excel_path: {excel_path}")  
    # 파일 확장자 확인
    file_ext = excel_path.suffix.lower()
    
    # 여러 Excel 라이브러리 시도 (우선순위 순)
    # 1. pandas + openpyxl/xlrd (가장 편리함)
    try:
        import pandas as pd
        
        # 모든 시트 읽기
        if file_ext == '.xlsx':
            excel_file = pd.ExcelFile(str(excel_path), engine='openpyxl')
        elif file_ext == '.xls':
            excel_file = pd.ExcelFile(str(excel_path), engine='xlrd')
        else:
            # 자동 감지
            excel_file = pd.ExcelFile(str(excel_path))
        
        sheets_text = {}
        for sheet_name in excel_file.sheet_names:
            df = pd.read_excel(excel_file, sheet_name=sheet_name)
            # DataFrame을 텍스트로 변환
            text_parts = []
            # 헤더 포함하여 모든 셀의 값을 문자열로 변환
            for idx, row in df.iterrows():
                row_values = [str(val) if pd.notna(val) else '' for val in row.values]
                text_parts.append(' | '.join(row_values))
            
            sheets_text[sheet_name] = '\n'.join(text_parts)

        print("using pandas")
        return sheets_text
    except ImportError as e:
        if 'pandas' in str(e):
            pass  # pandas가 없으면 다음 방법 시도
        elif 'openpyxl' in str(e) or 'xlrd' in str(e):
            # pandas는 있지만 엔진이 없는 경우
            raise ImportError(
                f"Excel 파일을 읽기 위한 엔진이 필요합니다.\n"
                f".xlsx 파일: pip install openpyxl\n"
                f".xls 파일: pip install xlrd"
            )
        else:
            raise
    
    # 2. openpyxl (xlsx 파일용)
    if file_ext == '.xlsx':
        try:
            from openpyxl import load_workbook
            
            workbook = load_workbook(str(excel_path), data_only=True)
            sheets_text = {}
            
            for sheet_name in workbook.sheetnames:
                sheet = workbook[sheet_name]
                text_parts = []
                
                for row in sheet.iter_rows(values_only=True):
                    row_values = [str(val) if val is not None else '' for val in row]
                    text_parts.append(' | '.join(row_values))
                
                sheets_text[sheet_name] = '\n'.join(text_parts)
            
            print("using openpyxl")
            return sheets_text
        except ImportError:
            pass
    
    # 3. xlrd (xls 파일용)
    if file_ext == '.xls':
        try:
            import xlrd
            
            workbook = xlrd.open_workbook(str(excel_path))
            sheets_text = {}
            
            for sheet_name in workbook.sheet_names():
                sheet = workbook.sheet_by_name(sheet_name)
                text_parts = []
                
                for row_idx in range(sheet.nrows):
                    row_values = [str(sheet.cell_value(row_idx, col_idx)) 
                                 for col_idx in range(sheet.ncols)]
                    text_parts.append(' | '.join(row_values))
                
                sheets_text[sheet_name] = '\n'.join(text_parts)
            
            print("using xlrd")
            return sheets_text
        except ImportError:
            pass
    
    # 모든 라이브러리가 없으면 에러
    raise ImportError(
        "Excel 텍스트 추출을 위한 라이브러리가 설치되지 않았습니다.\n"
        "다음 중 하나를 설치해주세요:\n"
        "  - pandas + openpyxl (권장): pip install pandas openpyxl\n"
        "  - pandas + xlrd (.xls 파일용): pip install pandas xlrd\n"
        "  - openpyxl (.xlsx 파일용): pip install openpyxl\n"
        "  - xlrd (.xls 파일용): pip install xlrd"
    )


def get_all_sheets_text(excel_path: str) -> str:
    """
    Excel 파일의 모든 시트 텍스트를 하나의 문자열로 반환하는 편의 함수
    
    Args:
        excel_path: Excel 파일 경로
    
    Returns:
        모든 시트의 텍스트를 합친 문자열
    """
    sheets_dict = extract_text_from_excel(excel_path)
    
    result_parts = []
    for sheet_name, sheet_text in sheets_dict.items():
        result_parts.append(f"=== 시트: {sheet_name} ===")
        result_parts.append(sheet_text)
        result_parts.append("")  # 빈 줄 추가
    
    return '\n'.join(result_parts)

In [ ]:
# Document Loader - 파일 형식에 따라 적절한 함수 호출

from pathlib import Path
from typing import Union

def load_document(file_path: str, password: str = None) -> str:
    """
    파일 형식에 따라 적절한 텍스트 추출 함수를 호출하여 텍스트를 반환하는 통합 함수
    
    지원 형식:
    - PDF: .pdf 파일 (암호화된 PDF 지원)
    - Excel: .xlsx, .xls 파일
    
    Args:
        file_path: 문서 파일 경로 (상대 경로 또는 절대 경로)
        password: PDF 파일이 암호화된 경우 비밀번호 (선택사항)
    
    Returns:
        추출된 텍스트 문자열
        - PDF: 전체 텍스트
        - Excel: 모든 시트의 텍스트를 합친 문자열
    
    Raises:
        FileNotFoundError: 파일을 찾을 수 없을 때
        ValueError: 지원하지 않는 파일 형식일 때 또는 PDF 암호가 틀렸을 때
        ImportError: 필요한 라이브러리가 설치되지 않았을 때
    """
    file_path_obj = Path(file_path)
    file_ext = file_path_obj.suffix.lower()
    
    # 파일 형식에 따라 적절한 함수 호출
    if file_ext == '.pdf':
        # PDF 파일 처리 (암호 전달)
        # return extract_text_from_pdf(file_path, password=password)
        return extract_text_from_pdf_with_docling(file_path, password)
    
    elif file_ext in ['.xlsx', '.xls']:
        # Excel 파일 처리 - 모든 시트의 텍스트를 하나의 문자열로 반환
        return get_all_sheets_text(file_path)
    
    else:
        raise ValueError(
            f"지원하지 않는 파일 형식입니다: {file_ext}\n"
            f"지원 형식: .pdf, .xlsx, .xls"
        )




In [ ]:
# 문서가 변액일임펀드 설정/해지 지시서인지 확인하는 LLM 노드 생성

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# 문서에서 추출한 텍스트가 변액일임펀드 설정/해지 지시서 여부를 판단하는 데이터 모델
class GradeDocument(BaseModel):
    """문서에서 추출한 텍스트가 변액일임펀드 설정/해지 지시서 여부를 판단하기 위한 이진 점수"""

    binary_score: str = Field(description="변액일임펀드 설정/해지 지시서 여부를 'yes' 또는 'no'로 판단합니다.")

# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")

# vLLM 모델 인스턴스 생성
llm = init_chat_model(
    "openai:",
    temperature=0.0,
    top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.1로 설정
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY
)

# GradeDocument 데이터 모델을 사용하여 구조화된 출력을 생성하는 LLM
structured_llm_grader = llm.with_structured_output(GradeDocument)

# 시스템 프롬프트 정의
system_prompt = """당신은 자산운용사에서 변액일임펀드 설정/해지 업무를 담당하는 오퍼레이터 입니다.
수익자가 보낸 문서가 변액일임펀드 설정/해지 지시서 인지 평가하세요.
문서에는 펀드에 대한 확정분 또는 청구분이 있어야 합니다.
확정분이 있을 경우 이에 대한 설정 금액 또는 해지 금액이 있어야 합니다.
청구분이 있을 경우 이에 대한 설정 금액 또는 해지 금액이 있어야 합니다.
문서의 변액일임펀드 설정/해지 지시서 여부를 나타내기 위해 이진 점수인 'yes' 또는 'no'를 부여하세요."""

# 프롬프트 템플릿 생성
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "수익자가 보낸 문서 내용: \n\n {original_text}")
])

# Retrieval 평가기 초기화
retrieval_grader = grade_prompt | structured_llm_grader

In [ ]:
# text 추출

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

_password = None
_password = '345678'

document_text = load_document(_document_file_path, _password)

print(f"   {document_text}")

In [ ]:
# document_text = "test"

relevance_result = retrieval_grader.invoke({"original_text": document_text})
print(relevance_result)

In [ ]:
# Graph 생성

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    file_path: str # 변액일임펀드 설정/해지 지시서 파일 경로
    password: str # 지시서 암호
    original_text: str # 지시서 원본 텍스트
    relevance_score: str # 지시서 여부 점수 : yes, no
    analysis_report: str # 지시서 분석 결과
    request_data: str # 지시서 청구분 데이터
    settlement_data: str # 지시서 확정분 데이터
    request_setup_data: str # 지시서 청구분 설정 데이터
    request_redemption_data: str # 지시서 청구분 해지 데이터
    settlement_setup_data: str # 지시서 확정분 설정 데이터
    settlement_redemption_data: str # 지시서 확정분 해지 데이터
    
graph_builder = StateGraph(State)

In [ ]:
# 지시서 문서 파일 로드 노드 생성

# 노드 - 문서(pdf, xls, xlsx) 로드
def node_load_document(state: State) -> State:
    file_path = state["file_path"]
    password = state["password"]
    original_text = load_document(file_path, password)
    return {"original_text": original_text}

graph_builder.add_node("load_document", node_load_document)

In [ ]:
# graph 엣지 설정

graph_builder.add_edge(START, "load_document")

graph_builder.add_edge("load_document", END)

In [ ]:
# graph 컴파일

graph = graph_builder.compile()

In [ ]:
# graph 실행

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"

_password = None
_password = '345678'
result = graph.invoke({"file_path": _document_file_path, "password": _password})

print(result)
print(f"   {result["original_text"]}")